In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile

PROJECT_ROOT = Path(".")

BIOGRID_DIR = PROJECT_ROOT / "Data_raw" / "Biogrid"
INTERIM_BIOGRID_DIR = PROJECT_ROOT / "Data_interim" / "biogrid"
NEG_DIR = PROJECT_ROOT / "Data_proc" / "negatives"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [INTERIM_BIOGRID_DIR, NEG_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BIOGRID_TXT = BIOGRID_DIR / "BIOGRID-ALL-4.4.248.tab2.txt"
BIOGRID_ZIP = BIOGRID_DIR / "BIOGRID-ALL-LATEST.tab2.zip"

E3_NEG_PATH = NEG_DIR / "negative_e3_initial_2x.csv"
DUB_NEG_PATH = NEG_DIR / "negative_dub_initial_2x.csv"
ALL_NEG_PATH = NEG_DIR / "negative_all_initial_2x.csv"

print("BioGRID txt exists:", BIOGRID_TXT.exists(), BIOGRID_TXT)
print("BioGRID zip exists:", BIOGRID_ZIP.exists(), BIOGRID_ZIP)
print("E3 neg exists:", E3_NEG_PATH.exists())
print("DUB neg exists:", DUB_NEG_PATH.exists())
print("ALL neg exists:", ALL_NEG_PATH.exists())

سلول ۲ — پیدا کردن فایل BioGRID و بررسی ستون‌ها

In [ ]:
def find_biogrid_table_path():
    """
    Prefer unzipped BioGRID txt.
    If not available, read first txt-like file from zip.
    """
    if BIOGRID_TXT.exists():
        return BIOGRID_TXT, None
    
    if BIOGRID_ZIP.exists():
        with zipfile.ZipFile(BIOGRID_ZIP, "r") as z:
            members = z.namelist()
            txt_members = [
                m for m in members
                if m.endswith(".txt") or m.endswith(".tab2.txt")
            ]
            if not txt_members:
                raise FileNotFoundError("No txt/tab2 file found inside BioGRID zip.")
            return BIOGRID_ZIP, txt_members[0]
    
    raise FileNotFoundError("No BioGRID txt or zip file found.")


biogrid_path, zip_member = find_biogrid_table_path()

print("BioGRID path:", biogrid_path)
print("Zip member:", zip_member)

# Read only header first
if zip_member is None:
    header_df = pd.read_csv(biogrid_path, sep="\t", nrows=0)
else:
    with zipfile.ZipFile(biogrid_path, "r") as z:
        with z.open(zip_member) as f:
            header_df = pd.read_csv(f, sep="\t", nrows=0)

print("Number of columns:", len(header_df.columns))
print("Columns:")
print(header_df.columns.tolist())

سلول ۳ — خواندن BioGRID فقط با ستون‌های لازم

In [ ]:
required_cols = [
    "#BioGRID Interaction ID",
    "Official Symbol Interactor A",
    "Official Symbol Interactor B",
    "Experimental System",
    "Experimental System Type",
    "Pubmed ID",
    "Organism Interactor A",
    "Organism Interactor B",
    "Score",
]

available_cols = set(header_df.columns)

missing = [c for c in required_cols if c not in available_cols]
if missing:
    raise ValueError(
        "Missing expected BioGRID columns:\n"
        + "\n".join(missing)
        + "\n\nAvailable columns:\n"
        + "\n".join(header_df.columns.tolist())
    )

if zip_member is None:
    biogrid = pd.read_csv(
        biogrid_path,
        sep="\t",
        dtype=str,
        usecols=required_cols,
        low_memory=False,
    )
else:
    with zipfile.ZipFile(biogrid_path, "r") as z:
        with z.open(zip_member) as f:
            biogrid = pd.read_csv(
                f,
                sep="\t",
                dtype=str,
                usecols=required_cols,
                low_memory=False,
            )

print("Raw BioGRID loaded:", biogrid.shape)
display(biogrid.head())

سلول ۴ — فیلتر انسان، physical interaction و ساخت edge key

In [ ]:
def normalize_gene_symbol(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    return x.upper()


def make_unordered_gene_pair_key(a, b):
    a = normalize_gene_symbol(a)
    b = normalize_gene_symbol(b)
    if pd.isna(a) or pd.isna(b):
        return np.nan
    if a == b:
        return np.nan
    return "|".join(sorted([a, b]))


bg = biogrid.copy()

n_raw = len(bg)

# Human only
bg = bg[
    (bg["Organism Interactor A"].astype(str).str.strip() == "9606") &
    (bg["Organism Interactor B"].astype(str).str.strip() == "9606")
].copy()

n_human = len(bg)

# Physical only
bg = bg[
    bg["Experimental System Type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("physical")
].copy()

n_human_physical = len(bg)

# Normalize genes
bg["geneA"] = bg["Official Symbol Interactor A"].map(normalize_gene_symbol)
bg["geneB"] = bg["Official Symbol Interactor B"].map(normalize_gene_symbol)

# Remove missing and self loops
bg = bg.dropna(subset=["geneA", "geneB"]).copy()
bg = bg[bg["geneA"] != bg["geneB"]].copy()

n_after_gene_clean = len(bg)

# Unordered key
bg["gene_pair_key"] = [
    make_unordered_gene_pair_key(a, b)
    for a, b in zip(bg["geneA"], bg["geneB"])
]

bg = bg.dropna(subset=["gene_pair_key"]).copy()

print("Raw BioGRID rows:", n_raw)
print("Human rows:", n_human)
print("Human physical rows:", n_human_physical)
print("After gene cleaning:", n_after_gene_clean)
print("Final physical rows:", len(bg))
print("Unique physical gene pairs:", bg["gene_pair_key"].nunique())

display(bg.head())

سلول ۵ — ساخت خروجی edge و evidence

In [ ]:
ppi_edges = (
    bg[["geneA", "geneB", "gene_pair_key"]]
    .drop_duplicates(subset=["gene_pair_key"])
    .reset_index(drop=True)
)

ppi_evidence = bg[[
    "#BioGRID Interaction ID",
    "geneA",
    "geneB",
    "gene_pair_key",
    "Experimental System",
    "Experimental System Type",
    "Score",
    "Pubmed ID",
]].rename(columns={
    "#BioGRID Interaction ID": "biogrid_interaction_id",
    "Experimental System": "experimental_system",
    "Experimental System Type": "experimental_system_type",
    "Score": "score",
    "Pubmed ID": "pmid",
})

ppi_edges.to_csv(
    INTERIM_BIOGRID_DIR / "biogrid_human_physical_edges.csv",
    index=False,
)

ppi_evidence.to_csv(
    INTERIM_BIOGRID_DIR / "biogrid_human_physical_evidence.csv",
    index=False,
)

biogrid_qc = pd.DataFrame([{
    "n_raw_rows": n_raw,
    "n_human_rows": n_human,
    "n_human_physical_rows": n_human_physical,
    "n_after_gene_clean": n_after_gene_clean,
    "n_final_physical_rows": len(bg),
    "n_unique_physical_gene_pair_key": ppi_edges["gene_pair_key"].nunique(),
}])

biogrid_qc.to_csv(
    QC_DIR / "biogrid_parse_qc.csv",
    index=False,
)

display(biogrid_qc)
display(ppi_edges.head())
display(ppi_evidence.head())

سلول ۶ — خواندن نگاتیوهای اولیه

In [ ]:
e3_neg = pd.read_csv(E3_NEG_PATH)
dub_neg = pd.read_csv(DUB_NEG_PATH)
negative_all = pd.read_csv(ALL_NEG_PATH)

print("E3 neg:", e3_neg.shape)
print("DUB neg:", dub_neg.shape)
print("ALL neg:", negative_all.shape)

display(e3_neg.head())
display(dub_neg.head())

سلول ۷ — فیلتر کردن نگاتیوها با PPI

In [ ]:
ppi_keys = set(ppi_edges["gene_pair_key"].dropna().astype(str))


def add_neg_gene_pair_key(df):
    out = df.copy()
    
    out["enz_gene_norm"] = out["enz_gene"].map(normalize_gene_symbol)
    out["sub_gene_norm"] = out["sub_gene"].map(normalize_gene_symbol)
    
    out["gene_pair_key"] = [
        make_unordered_gene_pair_key(a, b)
        for a, b in zip(out["enz_gene_norm"], out["sub_gene_norm"])
    ]
    
    return out


def filter_negative_by_ppi(neg_df, dataset_name):
    neg = add_neg_gene_pair_key(neg_df)
    
    neg["ppi_physical_flag"] = neg["gene_pair_key"].astype(str).isin(ppi_keys)
    
    removed = neg[neg["ppi_physical_flag"]].copy()
    kept = neg[~neg["ppi_physical_flag"]].copy()
    
    print("\n" + "="*80)
    print(dataset_name)
    print("Before:", len(neg))
    print("Removed by BioGRID physical PPI:", len(removed))
    print("Kept:", len(kept))
    
    return kept, removed


e3_neg_after_ppi, e3_removed_ppi = filter_negative_by_ppi(e3_neg, "E3")
dub_neg_after_ppi, dub_removed_ppi = filter_negative_by_ppi(dub_neg, "DUB")

negative_all_after_ppi = pd.concat(
    [e3_neg_after_ppi, dub_neg_after_ppi],
    ignore_index=True,
)

removed_all_ppi = pd.concat(
    [e3_removed_ppi, dub_removed_ppi],
    ignore_index=True,
)

print("\nALL")
print("Before:", len(negative_all))
print("Removed:", len(removed_all_ppi))
print("Kept:", len(negative_all_after_ppi))

display(removed_all_ppi.head())

سلول ۸ — QC فیلتر PPI

In [ ]:
def qc_after_ppi(before_df, after_df, removed_df, name):
    return {
        "dataset": name,
        "n_before": len(before_df),
        "n_removed_by_ppi": len(removed_df),
        "n_after": len(after_df),
        "removed_fraction": len(removed_df) / len(before_df) if len(before_df) else np.nan,
        "n_after_unique_pair_id": after_df["pair_id"].nunique(),
        "n_after_duplicate_pair_id_rows": int(after_df.duplicated("pair_id").sum()),
        "n_after_missing_enzyme_class": int(after_df["enzyme_class"].isna().sum()),
        "n_after_missing_enz_ac": int(after_df["enz_ac"].isna().sum()),
        "n_after_missing_sub_ac": int(after_df["sub_ac"].isna().sum()),
        "n_after_pair_id_starts_with_nan": int(after_df["pair_id"].astype(str).str.startswith("nan|").sum()),
        "n_removed_unique_gene_pair_key": removed_df["gene_pair_key"].nunique() if len(removed_df) else 0,
    }


ppi_filter_qc = pd.DataFrame([
    qc_after_ppi(e3_neg, e3_neg_after_ppi, e3_removed_ppi, "E3_negative_after_ppi"),
    qc_after_ppi(dub_neg, dub_neg_after_ppi, dub_removed_ppi, "DUB_negative_after_ppi"),
    qc_after_ppi(negative_all, negative_all_after_ppi, removed_all_ppi, "ALL_negative_after_ppi"),
])

display(ppi_filter_qc)

print("Label counts after PPI:")
print(negative_all_after_ppi["label"].value_counts(dropna=False))

print("Duplicate pair_id after PPI:", negative_all_after_ppi.duplicated("pair_id").sum())

ذخیره کن

In [ ]:
# Save filtered negatives
e3_neg_after_ppi.to_csv(
    NEG_DIR / "negative_e3_after_ppi_filter.csv",
    index=False
)

dub_neg_after_ppi.to_csv(
    NEG_DIR / "negative_dub_after_ppi_filter.csv",
    index=False
)

negative_all_after_ppi.to_csv(
    NEG_DIR / "negative_all_after_ppi_filter.csv",
    index=False
)

# Save removed negatives
e3_removed_ppi.to_csv(
    QC_DIR / "negative_e3_removed_by_ppi.csv",
    index=False
)

dub_removed_ppi.to_csv(
    QC_DIR / "negative_dub_removed_by_ppi.csv",
    index=False
)

removed_all_ppi.to_csv(
    QC_DIR / "negative_all_removed_by_ppi.csv",
    index=False
)

# Save QC
ppi_filter_qc.to_csv(
    QC_DIR / "ppi_filter_qc.csv",
    index=False
)

print("Saved:")
print(NEG_DIR / "negative_e3_after_ppi_filter.csv")
print(NEG_DIR / "negative_dub_after_ppi_filter.csv")
print(NEG_DIR / "negative_all_after_ppi_filter.csv")
print(QC_DIR / "negative_all_removed_by_ppi.csv")
print(QC_DIR / "ppi_filter_qc.csv")

In [ ]:
check_files = [
    NEG_DIR / "negative_e3_after_ppi_filter.csv",
    NEG_DIR / "negative_dub_after_ppi_filter.csv",
    NEG_DIR / "negative_all_after_ppi_filter.csv",
    QC_DIR / "negative_e3_removed_by_ppi.csv",
    QC_DIR / "negative_dub_removed_by_ppi.csv",
    QC_DIR / "negative_all_removed_by_ppi.csv",
    QC_DIR / "ppi_filter_qc.csv",
    INTERIM_BIOGRID_DIR / "biogrid_human_physical_edges.csv",
    INTERIM_BIOGRID_DIR / "biogrid_human_physical_evidence.csv",
]

for f in check_files:
    print(f.name, "exists:", f.exists())
    if f.exists() and f.suffix == ".csv":
        tmp = pd.read_csv(f)
        print("shape:", tmp.shape)